# Sky-View Factor

Sky-view factor (SVF) measures the fraction of the sky hemisphere visible from each cell in a digital elevation model. Values range from 0 (fully obstructed, e.g. a deep canyon) to 1 (flat open terrain with no horizon obstruction).

SVF is used in:
- **LiDAR archaeology** to reveal subtle terrain features hidden under canopy
- **Urban heat island studies** to quantify street canyon geometry
- **Solar energy modeling** as a proxy for diffuse sky irradiance
- **Terrain visualization** as an illumination-independent alternative to hillshade

The algorithm casts rays at evenly spaced azimuths from each cell and records the maximum elevation angle to the horizon along each ray. SVF is then:

$$\text{SVF} = 1 - \frac{1}{N}\sum_{d=1}^{N} \sin(\theta_d)$$

where $\theta_d$ is the maximum horizon angle in direction $d$.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial import sky_view_factor

## Generate synthetic terrain

We'll create a terrain surface with valleys, ridges, and a flat plateau to show how SVF responds to different landforms.

In [ ]:
# Create a 200x200 terrain with ridges and valleys
rows, cols = 200, 200
y = np.arange(rows, dtype=np.float64)
x = np.arange(cols, dtype=np.float64)
Y, X = np.meshgrid(y, x, indexing='ij')

# Sinusoidal ridges + a central peak
terrain = (
    40 * np.sin(X / 15) * np.cos(Y / 20)
    + 80 * np.exp(-((Y - 100)**2 + (X - 100)**2) / (2 * 30**2))
    + 20 * np.sin(Y / 10)
    + 200
)

dem = xr.DataArray(
    terrain,
    dims=['y', 'x'],
    coords={'y': np.arange(rows, dtype=float), 'x': np.arange(cols, dtype=float)},
    attrs={'res': (1.0, 1.0)},
)

fig, ax = plt.subplots(figsize=(8, 7))
dem.plot(ax=ax, cmap='terrain', cbar_kwargs={'label': 'Elevation'})
ax.set_title('Synthetic DEM')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Compute SVF with default parameters

The default uses `max_radius=10` cells and `n_directions=16` azimuth rays.

In [ ]:
svf = sky_view_factor(dem, max_radius=10, n_directions=16)

fig, ax = plt.subplots(figsize=(8, 7))
svf.plot(ax=ax, cmap='gray', vmin=0, vmax=1,
         cbar_kwargs={'label': 'Sky-View Factor'})
ax.set_title('SVF (max_radius=10, n_directions=16)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Effect of `max_radius`

A larger search radius captures more distant obstructions. This matters in terrain with broad features like wide valleys or distant ridgelines.

In [ ]:
radii = [5, 15, 30]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, r in zip(axes, radii):
    result = sky_view_factor(dem, max_radius=r, n_directions=16)
    result.plot(ax=ax, cmap='gray', vmin=0, vmax=1, add_colorbar=False)
    ax.set_title(f'max_radius={r}')
    ax.set_aspect('equal')

fig.suptitle('SVF at different search radii', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Effect of `n_directions`

More azimuth directions give a smoother result at the cost of longer computation. For most uses, 16 directions is a good balance.

In [ ]:
directions = [4, 16, 64]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, n in zip(axes, directions):
    result = sky_view_factor(dem, max_radius=15, n_directions=n)
    result.plot(ax=ax, cmap='gray', vmin=0, vmax=1, add_colorbar=False)
    ax.set_title(f'n_directions={n}')
    ax.set_aspect('equal')

fig.suptitle('SVF at different direction counts', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Comparison: SVF vs Hillshade

Hillshade depends on a specific sun angle, which creates directional bias. SVF provides omnidirectional illumination information, making features visible regardless of orientation.

In [ ]:
from xrspatial import hillshade

hs = hillshade(dem, azimuth=315, angle_altitude=45)
svf_result = sky_view_factor(dem, max_radius=15, n_directions=16)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

hs.plot(ax=axes[0], cmap='gray', add_colorbar=False)
axes[0].set_title('Hillshade (azimuth=315)')
axes[0].set_aspect('equal')

svf_result.plot(ax=axes[1], cmap='gray', vmin=0, vmax=1, add_colorbar=False)
axes[1].set_title('Sky-View Factor')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()